# 标签传播：让数据自己"说话"
> 难度标签：中级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：04_半监督学习 | 源文件：标签传播.py | 核心功能：图结构标签传播、LabelPropagation vs LabelSpreading、参数调优

## 概述

标签传播（Label Propagation）是基于图的半监督学习方法。核心思想：构建一个样本间的相似度图，有标签的样本带着自己的标签，通过图的边"传播"给无标签的邻居。迭代多次后，无标签样本被赋予最可能的标签。

这就像社交媒体上的信息扩散——一个有影响力的用户（有标签样本）发了一条消息（标签），他的关注者（相似样本）会看到并转发，层层传播下去。

脚本在 Iris 数据集上演示了标签传播，只用 15% 的有标签数据就能达到不错的分类效果。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.semi_supervised import LabelPropagation
from sklearn.metrics import accuracy_score, classification_report
# --- 导入库 ---
from sklearn.neighbors import KNeighborsClassifier

## 数学原理

### 1. 图构建与相似度矩阵

**代码对应**：`LabelPropagation(kernel="knn", n_neighbors=7)` 构建相似度图。

构建 $n \times n$ 相似度矩阵 $\mathbf{W}$：

**RBF 核**：$W_{ij} = \exp(-\gamma\|\mathbf{x}_i - \mathbf{x}_j\|^2)$

**KNN 核**：$W_{ij} = 1$ 如果 $\mathbf{x}_j$ 是 $\mathbf{x}_i$ 的 $k$ 近邻之一，否则 $W_{ij} = 0$。

### 2. 标签传播的迭代公式

设 $\mathbf{F} \in \mathbb{R}^{n \times K}$ 为标签矩阵（$K$ 为类别数），$\mathbf{Y}$ 为初始标签（有标签位置为 one-hot，无标签为 0）。

**归一化**：$\mathbf{S} = \mathbf{D}^{-1/2}\mathbf{W}\mathbf{D}^{-1/2}$（对称归一化）

**迭代更新**：$\mathbf{F}^{(t+1)} = \alpha\mathbf{S}\mathbf{F}^{(t)} + (1-\alpha)\mathbf{Y}$

收敛后：$\mathbf{F}^* = (\mathbf{I} - \alpha\mathbf{S})^{-1}(1-\alpha)\mathbf{Y}$

### 3. LabelPropagation vs LabelSpreading

**LabelPropagation**：$\alpha \to 1$，标签完全沿图传播，可能不稳定。

**LabelSpreading**：$\alpha \in (0, 1)$（`alpha=0.2` 常用），保留部分原始标签信息，更鲁棒。

**代码对应**：`LabelSpreading(alpha=0.2)` 中的 alpha 控制正则化强度。

### 4. 收敛性保证

标签传播算法在 $\alpha < 1$ 时保证收敛到唯一不动点。这是因为 $\alpha\mathbf{S}$ 的谱半径 $\rho(\alpha\mathbf{S}) < 1$（$\mathbf{S}$ 的最大特征值为 1），迭代序列是压缩映射。

### 5. 计算复杂度

- 构建 $\mathbf{W}$：$O(n^2 p)$（RBF）或 $O(np\log n)$（KNN + 空间索引）
- 每次迭代：$O(n^2 K)$（矩阵乘法）
- 总复杂度：$O(Tn^2K)$，$T$ 为迭代次数

### 1. 加载数据

首先加载数据集，为后续训练和评估做准备。

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 只保留少量标签（约 15%）
np.random.seed(42)
n_labeled = int(len(y_train) * 0.15)
labeled_idx = np.random.choice(len(y_train), n_labeled, replace=False)
y_train_semi = np.full_like(y_train, fill_value=-1)
y_train_semi[labeled_idx] = y_train[labeled_idx]

print(f"=== 数据划分 ===")
print(f"训练集: {len(X_train)} 样本 (有标签: {(y_train_semi >= 0).sum()}, "
      f"无标签: {(y_train_semi == -1).sum()})")

### 2. 基准：仅用有标签数据

运行 2. 基准：仅用有标签数据 的代码，观察算法在该环节的行为。

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train[labeled_idx], y_train[labeled_idx])
print(f"仅用有标签数据 KNN: {knn.score(X_test, y_test):.4f}")

### 3. 标签传播算法

运行 3. 标签传播算法 的代码，观察算法在该环节的行为。

In [ ]:
# kernel: 'knn' 或 'rbf'
# gamma: RBF 核参数
# n_neighbors: KNN 核的邻居数
lp = LabelPropagation(kernel="knn", n_neighbors=7, gamma=15, max_iter=1000)
lp.fit(X_train, y_train_semi)

y_pred = lp.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\n=== 标签传播 (LabelPropagation) ===")
print(f"测试准确率: {acc:.4f}")
label_distributions = lp.label_distributions_
# --- 输出结果 ---
print(f"标签传播矩阵形状: {label_distributions.shape}")
print(f"各类别分配概率（前 5 个样本）:")
print(label_distributions[:5].round(3))

### 4. 不同参数对比

探索关键超参数对模型行为的影响。

In [ ]:
print("\n=== n_neighbors 对比 ===")
for nn in [3, 5, 7, 10, 15, 30]:
    lp_n = LabelPropagation(kernel="knn", n_neighbors=nn, max_iter=1000)
    lp_n.fit(X_train, y_train_semi)
    acc = lp_n.score(X_test, y_test)
# --- 输出结果 ---
    print(f"  n_neighbors={nn:>2}: 测试准确率={acc:.4f}")

### 5. RBF 核 vs KNN 核

运行 5. RBF 核 vs KNN 核 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 核函数对比 ===")
for kernel, params in [("knn", {"n_neighbors": 7}), ("rbf", {"gamma": 15})]:
    lp_k = LabelPropagation(kernel=kernel, max_iter=1000, **params)
    lp_k.fit(X_train, y_train_semi)
    acc = lp_k.score(X_test, y_test)
# --- 输出结果 ---
    print(f"  kernel={kernel:>3}: 测试准确率={acc:.4f}")

### 6. LabelPropagation vs LabelSpreading

运行 6. LabelPropagation vs LabelSpreading 的代码，观察算法在该环节的行为。

In [ ]:
from sklearn.semi_supervised import LabelSpreading
print("\n=== LabelPropagation vs LabelSpreading ===")
for name, model in [
    ("LabelPropagation", LabelPropagation(kernel="knn", n_neighbors=7, max_iter=1000)),
    ("LabelSpreading", LabelSpreading(kernel="knn", n_neighbors=7, alpha=0.2, max_iter=1000)),
# --- 继续 ---
]:
    model.fit(X_train, y_train_semi)
    acc = model.score(X_test, y_test)
    n_iter = model.n_iter_
    print(f"  {name:>20}: 测试准确率={acc:.4f}, 迭代次数={n_iter}")

### 7. 标签传播原理

运行 7. 标签传播原理 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 标签传播原理 ===")
print("1. 构建相似度图（每个样本是节点，边权 = 相似度）")
print("2. 标签沿边传播：每个节点的标签 = 邻居标签的加权平均")
print("3. 迭代直到收敛")
print("4. 最终每个节点的标签 = 收敛后的概率分布中最大的类别")

print("\n=== LabelPropagation vs LabelSpreading ===")
print("LabelPropagation: 标签完全转移（归一化到 [0,1]），可能不稳定")
print("LabelSpreading: 用 alpha 参数正则化，保留部分原始标签信息，更稳定")

print("\n=== 标签传播要点 ===")
print("- 优点：无需训练显式模型，利用数据几何结构")
print("- 适合：数据有明显的聚类结构，同类样本距离近")
print("- 对图构建参数（kernel, gamma, n_neighbors）敏感")
print("- 计算复杂度 O(n²)，大数据集较慢")

## 关键代码解释

### 无标签数据用 -1 标记

```python
y_train_semi = np.full_like(y_train, fill_value=-1)
y_train_semi[labeled_idx] = y_train[labeled_idx]
```

sklearn 的半监督 API 约定：-1 表示无标签样本。这个约定贯穿 LabelPropagation、LabelSpreading 和 SelfTrainingClassifier。

### LabelPropagation vs LabelSpreading

```python
LabelPropagation(kernel="knn", n_neighbors=7)
LabelSpreading(kernel="knn", n_neighbors=7, alpha=0.2)
```

- **LabelPropagation**：标签完全沿边传播，可能数值不稳定
- **LabelSpreading**：用 alpha 参数正则化，保留部分原始标签信息，更稳定。alpha=0.2 意味着 80% 来自传播，20% 来自原始标签

## 使用示例

```python
from sklearn.semi_supervised import LabelPropagation
lp = LabelPropagation(kernel="knn", n_neighbors=7)
lp.fit(X, y_semi)  # y_semi 中 -1 表示无标签
print(lp.predict(X_test))
```

## 注意事项

1. **O(n²) 计算量**：构建相似度矩阵需要计算所有点对的距离
2. **对 kernel 参数敏感**：knn 和 rbf 效果可能差异很大
3. **需要足够的有标签样本启动传播**
4. **数据需要有聚类/流形结构**：同类样本距离近时效果好

## 总结与延伸

以上代码展示了 **标签传播** 的完整流程。

**解读要点**：
- 关注 **标签传播** 的准确率随迭代的变化
- 对比有监督 vs 半监督的性能差距
- 观察少量标签下的学习效果

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **图神经网络（GNN）**：标签传播的"深度学习版本"，如 GraphSAGE、GAT
- **标签传播的理论保证**：在流形假设下，标签传播的收敛性有严格数学证明
- **主动学习 + 标签传播**：用标签传播识别最有信息量的未标注样本，优先标注
